# Practice 2 — Sweep fine-tune trên GPU Colab

Sweep 2 siêu tham số đúng yêu cầu đề bài:
- learning rate: 5 giá trị {0.0001, 0.0005, 0.001, 0.003, 0.005}
- batch-size: {32, 64}
- 3 mô hình: resnet18, vgg11, densenet121 — strategy fine_tune, optimizer adam
- Tổng 3 x 5 x 2 = 30 cấu hình, mỗi cái 3 epoch, ảnh 64.

Chạy từng ô theo thứ tự từ trên xuống dưới.

## 1. Kiểm tra GPU (phải hiện T4 mới đúng)

In [ ]:
import torch
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

## 2. Lấy code từ GitHub

In [ ]:
REPO_URL = "https://github.com/Sper-il/Lab2.git"

import os
if not os.path.isdir("/content/Lab2"):
    !git clone $REPO_URL /content/Lab2
else:
    !git -C /content/Lab2 pull
%cd /content/Lab2
!git log --oneline -3
!git status --short

## 3. Cài thư viện

In [ ]:
!pip -q install tensorboard pyyaml matplotlib pillow numpy
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)

## 4. Tải CIFAR-10 về máy Colab

In [ ]:
!python scripts/download_data.py
!python scripts/verify_data.py

## 5. Chạy nhanh 1 cấu hình thử máy (1 epoch)

In [ ]:
!python -m src.training --config configs/train_config.yaml --models resnet18 --lr 0.001 --batch-size 32 --epochs 1 --image-size 64 --strategy fine_tune --optimizer adam --no-tensorboard

## 6. Chạy sweep 30 cấu hình (lâu nhất — cứ để chạy)

In [ ]:
!python -m src.training --config configs/train_config.yaml --models resnet18 vgg11 densenet121 --lr 0.0001 0.0005 0.001 0.003 0.005 --batch-size 32 64 --epochs 5 --image-size 64 --strategy fine_tune --optimizer adam

## 7. Đánh giá test cho từng run mới (bỏ qua run cũ kéo về từ GitHub)

In [ ]:
import glob, os, subprocess, time
cutoff = time.time() - 24 * 3600  # chỉ đánh giá run mới trong 24 giờ qua
for ckpt in sorted(glob.glob("results/runs/*/resnet18-*/checkpoints/best.pt")) + sorted(glob.glob("results/runs/*/vgg11-*/checkpoints/best.pt")) + sorted(glob.glob("results/runs/*/densenet121-*/checkpoints/best.pt")):
    if os.path.getmtime(ckpt) < cutoff:
        print("bo qua run cu:", ckpt)
        continue
    print("=", ckpt)
    subprocess.run(["python", "-m", "src.evaluation", "--checkpoint", ckpt], check=True)

## 8. Nén kết quả để tải về

In [ ]:
import glob, time
new_runs = [d for d in glob.glob("results/runs/*") if (time.time() - os.path.getmtime(d)) < 24 * 3600]
print("run moi:", new_runs)
!tar -czf /content/sweep-results.tar.gz $new_runs
print("Tải file /content/sweep-results.tar.gz ở panel Files bên trái, giải nén vào results/runs/ trong máy bạn.")